Warehouse Environment

In [1]:
class Warehouse:
    def __init__(self, num_sections):
        """
        Initialize the warehouse.

        Each section stores a list of items.
        Example:
        section 0 → ['item1', 'item2']
        section 1 → []
        """
        self.num_sections = num_sections

        # Dictionary: section → list of items
        self.sections = {i: [] for i in range(num_sections)}

    def store_item(self, section, item):
        """
        Store an item in a given section.

        :param section: Section number
        :param item: Item name
        """
        if section in self.sections:
            self.sections[section].append(item)

    def retrieve_item(self, section, item):
        """
        Retrieve (remove) an item from a section.

        :return: True if successful, False otherwise
        """
        if section in self.sections and item in self.sections[section]:
            self.sections[section].remove(item)
            return True
        return False

Task Defination

In [2]:
class Task:
    def __init__(self, task_type, location, item):
        """
        Define a task for robots.

        :param task_type: 'retrieve' or 'restock'
        :param location: (section, aisle)
        :param item: item involved
        """
        self.task_type = task_type
        self.location = location
        self.item = item

    def __repr__(self):
        """
        String representation (useful for debugging)
        """
        return f"Task(type={self.task_type}, location={self.location}, item={self.item})"

Robot Agent

In [3]:
class Robot:
    def __init__(self, warehouse):
        """
        Initialize robot (AI agent).

        Attributes:
        - position → current location
        - task → assigned task
        - reward → performance score
        """
        self.warehouse = warehouse
        self.position = (0, 0)  # (section, aisle)
        self.task = None
        self.reward = 0

    def set_position(self, section, aisle):
        """Set robot position."""
        self.position = (section, aisle)

    def assign_task(self, task):
        """Assign a task to robot."""
        self.task = task

    def distance_to(self, location):
        """
        Calculate Manhattan Distance:
        |x1 - x2| + |y1 - y2|
        """
        return abs(self.position[0] - location[0]) + \
               abs(self.position[1] - location[1])

    def reassign_task(self):
        """Remove current task (used in conflict resolution)."""
        self.task = None

    def complete_task(self):
        """
        Execute the assigned task.

        Steps:
        1. Move to task location
        2. Perform action (retrieve/restock)
        3. Return success/failure
        """
        if not self.task:
            print("No task assigned.")
            return False

        task_type = self.task.task_type
        location = self.task.location
        item = self.task.item

        # Step 1: Move to location
        self.set_position(*location)

        # Step 2: Perform task
        if task_type == 'retrieve':
            success = self.warehouse.retrieve_item(location[0], item)

            if success:
                print(f"[SUCCESS] Robot at {self.position} retrieved {item}")
            else:
                print(f"[FAILED] Robot at {self.position} could not retrieve {item}")
                return False

        elif task_type == 'restock':
            self.warehouse.store_item(location[0], item)
            print(f"[SUCCESS] Robot at {self.position} restocked {item}")

        # Step 3: Clear task
        self.task = None
        return True

Task Allocation strategy

In [4]:
def allocate_task(task, robots):
    """
    Assign task to the nearest available robot.

    Strategy:
    - Ignore busy robots
    - Choose robot with minimum distance
    """

    # Filter only free robots
    available_robots = [r for r in robots if r.task is None]

    if not available_robots:
        print("No available robots!")
        return

    # Select nearest robot
    nearest_robot = min(
        available_robots,
        key=lambda r: r.distance_to(task.location)
    )

    nearest_robot.assign_task(task)

Conflict Resolution

In [5]:
def resolve_conflict(robot1, robot2):
    """
    Resolve conflict when two robots have same task.

    Rule:
    - Robot closer to task keeps it
    - Other robot loses task
    """

    if robot1.task and robot2.task and robot1.task == robot2.task:

        if robot1.distance_to(robot1.task.location) < robot2.distance_to(robot2.task.location):
            robot2.reassign_task()
        else:
            robot1.reassign_task()

Reward System

In [6]:
def reward_robot(robot, success):
    """
    Reward mechanism:

    +10 → successful task
    -5  → failed task
    """

    if success:
        robot.reward += 10
    else:
        robot.reward -= 5

Simulation / Main Execution

In [8]:
def main():
    # Create warehouse
    warehouse = Warehouse(num_sections=5)

    # Add initial inventory
    warehouse.store_item(3, "Widget A")

    # Create robots
    robots = [Robot(warehouse) for _ in range(5)]

    # Assign initial positions
    for i, robot in enumerate(robots):
        robot.set_position(i, 0)

    # Create task
    task = Task('retrieve', (3, 2), 'Widget A')

    # Allocate task
    allocate_task(task, robots)

    # Example conflict scenario
    duplicate_task = Task('restock', (2, 3), 'Widget B')
    robots[0].assign_task(duplicate_task)
    robots[1].assign_task(duplicate_task)

    resolve_conflict(robots[0], robots[1])

    # Execute tasks and reward robots
    for robot in robots:
        success = robot.complete_task()
        reward_robot(robot, success)

    # Display final rewards
    print("\n--- Robot Rewards ---")
    for i, robot in enumerate(robots):
        print(f"Robot {i}: {robot.reward}")


# Run program
main()

No task assigned.
[SUCCESS] Robot at (2, 3) restocked Widget B
No task assigned.
[SUCCESS] Robot at (3, 2) retrieved Widget A
No task assigned.

--- Robot Rewards ---
Robot 0: -5
Robot 1: 10
Robot 2: -5
Robot 3: 10
Robot 4: -5
